# 11.5 哪些文字層需要一起調整？


只訓練接頭改動小，但有時語言模型還需要適應新的視覺上下文。我們可以開放最後一層，也可以開放所有參數。選擇範圍像修一間房子：只改門把、改一面牆、全部翻修，成本和影響不同。本節先算各方案會動多少數字，不用成本數值替代效果比較。

前置是[11.2參數凍結與優化器](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)及[4.5一個模型區塊](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)。區塊block是一次注意力與特徵處理等運算組成的層；最後一個block最靠近輸出。partial指本例只開放它和圖片接頭，all則開放整個包裝模型，包括語言、視覺與聲音部分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.multimodal import MultiModalLM

model = MultiModalLM(TinyLM(ModelConfig(width=8)))
for mode in ["projector", "partial", "all"]:
    model.requires_grad_(mode == "all")
    model.image_projector.requires_grad_(True)
    if mode == "partial":
        model.language.blocks[-1].requires_grad_(True)
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(mode, count)

輸入同一個寬度8模型，每輪先重設全部開關，再開放該方案需要的部分，避免上輪的狀態殘留。`[-1]`選最後一個block，`numel`數所有允許梯度的數字。當前預設配置輸出projector 136、partial 976、all 8296。最後一個數量包括未在圖片任務中使用的聲音參數；允許梯度不等於每筆都會收到梯度，這點仍需按[11.2](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)檢查實際路徑。

這些數量說明訓練成本起點。以32位元浮點數保存，每個參數本身約4位元組；還要加梯度與優化器狀態，AdamW通常另外保存兩組歷史數字，因此訓練記憶體不只參數本身。各參數組是否真的分配狀態取決於有沒有收到梯度，不能機械用一個倍數報告所有實際用量。

更多自由度可能幫助新接口，也可能把原來的文字能力改變。公平比較需同樣訓練資料、總有效回答位置或時間預算，並另測純文字任務。[有效回答位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.5.html)指目標標籤未被忽略、實際計入猜錯代價的位置，問句與補齊位置通常不計入。例如一份例子有3個計入代價的回答位置，用它訓練10次就提供30個位置；控制這個總數，有助於比較方案是否接受相近的學習量。只看到all參數多，不足以斷言它最好；只看到projector便宜，也不代表它一定夠用。

partial在這個預設模型裡只開放一個語言block，所以976減136=840，是新增語言區塊的數字數，不是新增訓練樣本數。若語言模型有四層，最後一層仍只有一層參數；all則開放更多層。訓練開始前還要用實際輸入檢查梯度：all名義開放聲音編碼器，純圖片批次沒有波形，它不一定收到訓練信號。參數統計提供配置成本，實際更新記錄提供發生事實，兩者並排才不會把沒有用到的模塊說成也學了新任務。

練習把ModelConfig加`layers=2`，先預測projector仍136，因為寬度沒變；partial仍只開最後一層，all則多一個語言block。重跑核對，再說出為什麼層數增加沒有自動讓接頭矩陣變大。這個對照把寬度、深度與可訓練範圍三個不同選擇分開了。
